# Road Network Visualization
This notebook visualizes the relationship between the **KMZ Backbones** (`query.kmz`) and the **Processed Road Segments** (`integrated_road_network.parquet`).

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from shapely import wkb
import os
import zipfile
from fiona.drvsupport import supported_drivers

# Enable KML support in fiona
supported_drivers["KML"] = "rw"
supported_drivers["LIBKML"] = "rw"

## 1. Load data 

### 1.1 load Road Segments
Loading from `data/processed/integrated_road_network.parquet`. The geometry is stored as WKB bytes, so we loads it into Shapely objects.

In [ ]:
segments_path = "../data/processed/integrated_road_network.parquet"

gdf_segments = gpd.read_parquet(segments_path)
print(gdf_segments.geometry.crs)

### 1.2. Load KMZ Backbones
Opening `data/raw/roads/query.kmz` and reading the internal `doc.kml`.

In [ ]:
small_segment_length_m = 2000

kmz_path="../data/raw/roads/query.kmz"
gdf_backbone = gpd.read_file(kmz_path, driver='KML')
print(f'initial crs: {gdf_backbone.geometry.crs}')
gdf_backbone = gdf_backbone.to_crs(gdf_segments.crs)
print(f'transformed crs for length calc: {gdf_backbone.geometry.crs}')
gdf_backbone['backbone_length_m'] = gdf_backbone.geometry.length
gdf_backbone = gdf_backbone.to_crs(4326)
print(f'final crs for length calc: {gdf_backbone.geometry.crs}')

print(f"Loaded {len(gdf_backbone)} backbone paths.")

In [ ]:
gdf_backbone_filtered = gdf_backbone[gdf_backbone['backbone_length_m'] < small_segment_length_m].copy()

## 2. Visualizations

In [ ]:

m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles="CartoDB positron")

# Add Backbones Layer
folium.GeoJson(
    gdf_backbone,
    name="KMZ Backbones",
    style_function=lambda x: {"color": "blue", "weight": 5, "opacity": 0.8}
).add_to(m)
m

In [ ]:

m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles="CartoDB positron")

# Add Backbones Layer
folium.GeoJson(
    gdf_backbone_filtered,
    name="KMZ Backbones",
    style_function=lambda x: {"color": "blue", "weight": 5, "opacity": 0.8}
).add_to(m)
m

In [ ]:
m = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles="CartoDB positron")

# Add Backbones Layer

folium.GeoJson(
    gdf_segments_4326,
    name=f"Road Segments",
    style_function=lambda x: {"color": "red", "weight": 2, "opacity": 0.6}
).add_to(m)

m